In [3]:
import pandas as pd
import numpy as np

In [4]:
# Load data
df = pd.read_csv("SphereSleep.sensor_data.csv")

In [5]:
# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")
df = df[df['timestamp'].dt.year > 1970]

In [6]:
# Create session_date
df["session_date"] = df["timestamp"].dt.date
df.loc[df["timestamp"].dt.hour < 6, "session_date"] = (
    df.loc[df["timestamp"].dt.hour < 6, "timestamp"] - pd.Timedelta(days=1)
).dt.date

In [7]:
# Create sleep_session
df["sleep_session"] = (
    df["user_id"].astype(str) + "_" + df["session_date"].astype(str)
)

## Thermal Anomaly Label

In [8]:
def classify_thermal_comfort(row):
    """
    Classify thermal comfort based on temperature and humidity
    Returns: 'Normal' (0) or 'Abnormal' (1)
    """
    temp = row['temperature']
    humidity = row['humidity']
    
    # Optimal ranges
    temp_optimal = 24 <= temp <= 27
    humidity_optimal = 40 <= humidity <= 60
    
    # Abnormal conditions
    if temp < 22 or temp > 28:  # Critical temperature
        return 1
    if humidity < 30 or humidity > 70:  # Critical humidity
        return 1
    if not temp_optimal and not humidity_optimal:  # Both suboptimal
        return 1
    
    return 0  # Normal

In [9]:
df["thermal_anomaly"] = df.apply(classify_thermal_comfort, axis=1)

In [10]:
# Select relevant columns
df = df[['timestamp', 'user_id', '_id', 'temperature', 'humidity', 
         'light_intensity', 'sound_level', 'session_date', 'thermal_anomaly']]

In [11]:
# Save
df.to_csv("output_with_thermal_anomaly.csv", index=False)

In [12]:
print(f"Total records: {len(df)}")
print(f"Normal: {(df['thermal_anomaly']==0).sum()}")
print(f"Abnormal: {(df['thermal_anomaly']==1).sum()}")
print(f"Class balance: {df['thermal_anomaly'].value_counts(normalize=True)}")

Total records: 61315
Normal: 41241
Abnormal: 20074
Class balance: thermal_anomaly
0    0.672609
1    0.327391
Name: proportion, dtype: float64
